<div style="background:linear-gradient(135deg,#312E81,#7E22CE,#BE123C);border-radius:24px;padding:44px 48px;color:white">
"
        "<div style="font-size:14px;letter-spacing:0.12em">BRANDCAST · DATA COLLECTION</div>
"
        "<h1 style="margin:10px 0;font-size:36px">RACOI 연간 자료 수집</h1>
"
        "<p style="margin:0;font-size:17px">공개 원본의 기간·구조·수집 상태를 재현 가능한 표로 확인합니다.</p>
"
        "</div>

<div style="background:#F5F3FF;border-left:6px solid #7C3AED;border-radius:14px;padding:22px 26px">
"
        "<h2 style="margin:0;color:#312E81">핵심 질문</h2>
"
        "<p style="margin:10px 0 0;color:#111827;font-size:17px">최근 1년 동안 어떤 RACOI 공개 자료를 어느 기간까지 확보했으며, 실제 응답 구조와 누락·제약은 무엇인가?</p>
"
        "</div>

## 1. 목적과 범위

이 노트북은 이슈 #3 수집 모듈이 남긴 원본과 `수집_메타데이터.jsonl`을 읽어 수집 범위·실패·중복·파일 구조를 표로 확인한다. 인터넷반응은 주간 XLSX, 종합반응은 월간 XLSX, 댓글반응은 프로그램·주차별 공개 JSON 집계다.

원본을 수정하거나 MariaDB에 적재하지 않으며, 결측값 대체·분석 지표·후보 순위·Q1~Q6 그래프도 만들지 않는다. 미수집은 실제 0과 구분해 `미수집`으로 표시한다.

In [1]:
# 목적: 프로젝트 경로와 원본·메타데이터 위치를 준비한다.
# 입력: 프로젝트 루트의 데이터/원본 폴더와 선택적 .env 파일.
# 출력: 이후 표에 사용하는 경로, 환경변수와 표시 도우미.
# 기간: 노트북 실행 시점의 로컬 수집 이력을 그대로 사용한다.
# NULL/0 처리: 파일·메타데이터가 없으면 0으로 바꾸지 않고 상태 메시지로 표시한다.
from __future__ import annotations

import json
import os
import subprocess
import sys
from collections import Counter
from datetime import date, datetime, timedelta
from pathlib import Path

import pandas as pd
from IPython.display import HTML, display
from openpyxl import load_workbook

ROOT = Path.cwd().resolve()
if not (ROOT / "수집").exists():
    ROOT = ROOT.parent
RAW_ROOT = ROOT / "데이터" / "원본"
METADATA_PATH = RAW_ROOT / "수집_메타데이터.jsonl"

def show_table(rows, *, title, columns=None, limit=20):
    frame = pd.DataFrame(rows, columns=columns)
    display(HTML(f"<h3 style='color:#312E81;margin-top:22px'>{title}</h3>"))
    display(HTML(f"<p style='color:#4B5563'>표시 행 수: {min(len(frame), limit)} / 전체 행 수: {len(frame)}</p>"))
    display(frame.head(limit) if len(frame) else pd.DataFrame([{"상태": "원본 없음 또는 미수집"}]))
    return frame

def read_metadata():
    if not METADATA_PATH.exists():
        return []
    rows = []
    for line in METADATA_PATH.read_text(encoding="utf-8").splitlines():
        try:
            value = json.loads(line)
        except json.JSONDecodeError:
            continue
        if isinstance(value, dict):
            rows.append(value)
    return rows

metadata_rows = read_metadata()
print(f"프로젝트 루트: {ROOT}")
print(f"수집 이력: {METADATA_PATH} ({len(metadata_rows)}건)")

프로젝트 루트: C:\Users\yooha\OneDrive\Desktop\git-hub-projects\Racoi\BrandCast
수집 이력: C:\Users\yooha\OneDrive\Desktop\git-hub-projects\Racoi\BrandCast\데이터\원본\수집_메타데이터.jsonl (101건)


## 2. 실행 매개변수와 기간 기준

수집 실행은 기본적으로 꺼져 있다. 기존 원본과 이력을 검토할 때는 `RUN_COLLECTION = False`를 유지한다. 새 시범 수집은 `True`로 바꾼 뒤 `PROBE_MODE = True`로 실행한다. 연간 수집은 시작일·종료일을 명시하고, RACOI가 제공하는 최신 완료 주차·월을 넘지 않는다.

In [2]:
# 목적: 수집 실행 여부와 기간 매개변수를 명시적으로 표시한다.
# 입력: RUN_COLLECTION, PROBE_MODE, START_DATE, END_DATE, COMMENT_PJSEQ.
# 출력: 선택한 실행 모드와 수집기 JSON 결과.
# 기간: probe는 최신 완료 1주·1개월, range는 지정한 시작일·종료일이다.
# NULL/0 처리: 수집 미실행은 실패나 0건이 아니라 기존 이력 조회 상태로 표시한다.
RUN_COLLECTION = False
PROBE_MODE = True
START_DATE = os.getenv("DATA_START_DATE") or None
END_DATE = os.getenv("DATA_END_DATE") or None
COMMENT_PJSEQ = 11487

command = [sys.executable, "수집/수집_실행.py", "--comment-pjseq", str(COMMENT_PJSEQ)]
if PROBE_MODE:
    command.append("--probe")
elif START_DATE and END_DATE:
    command.extend(["--start-date", START_DATE, "--end-date", END_DATE])

execution_result = None
if RUN_COLLECTION:
    completed = subprocess.run(command, cwd=ROOT, text=True, capture_output=True, check=False)
    execution_result = json.loads(completed.stdout) if completed.stdout.strip() else {"status": "failed", "error": completed.stderr}
    metadata_rows = read_metadata()
    show_table([execution_result.get("execution", {})], title="이번 실행 매개변수", limit=1)
else:
    display(HTML("<div style='background:#FFFBEB;border:1px solid #FDE68A;border-radius:14px;padding:16px'><b>주의 · 수집 미실행</b><br>기존 원본과 JSONL 수집 이력만 조회합니다. 새 공개 요청은 보내지 않았습니다.</div>"))
    show_table([{"실행 모드": "기존 이력 조회", "수집 명령": " ".join(command), "시작일": START_DATE or "환경변수 미설정", "종료일": END_DATE or "환경변수 미설정"}], title="노트북 실행 설정", limit=1)

,실행 모드,수집 명령,시작일,종료일
0,기존 이력 조회,C:\Users\yooha\OneDrive\Desktop\git-hub-projec...,환경변수 미설정,환경변수 미설정


## 3. 수집 실행 요약과 파일 목록

`success`는 새 원본 저장, `duplicate`는 기존과 같은 원본을 재사용한 정상 재실행, `failed`는 실패 사유가 기록된 시도다. 파일 경로가 실제로 존재하지 않으면 성공 상태라도 `누락 파일`로 분리한다.

In [3]:
# 목적: 수집 상태와 실제 원본 파일 존재 여부를 함께 확인한다.
# 입력: 수집_메타데이터.jsonl과 원본 파일 경로.
# 출력: 상태별 건수, 실행 요약표, 원천별 파일 목록표.
# 기간: 메타데이터 행에 기록된 period_start·period_end를 그대로 표시한다.
# NULL/0 처리: 실패와 누락 파일은 0건으로 대체하지 않고 사유를 표시한다.
status_counts = Counter(row.get("collection_status", "unknown") for row in metadata_rows)
show_table([
    {"상태": status, "시도 수": status_counts.get(status, 0)}
    for status in ("success", "duplicate", "failed")
], title="수집 실행 상태 요약", limit=3)

summary_rows = []
file_rows = []
for row in metadata_rows:
    raw_path = row.get("file_path")
    exists = bool(raw_path) and (ROOT / Path(str(raw_path).replace("\\", "/"))).is_file()
    summary_rows.append({
        "원천": row.get("source_name"), "자료유형": row.get("dataset_name"),
        "기간 시작": row.get("period_start"), "기간 종료": row.get("period_end"),
        "pjseq": row.get("request_pjseq"), "주차": row.get("request_nweek"),
        "상태": row.get("collection_status"), "HTTP": row.get("http_status"),
        "형식": row.get("response_format"), "행·레코드 수": row.get("row_count"),
        "실패 사유": row.get("error_message") or "",
    })
    file_rows.append({
        "원천": row.get("source_name"), "자료유형": row.get("dataset_name"),
        "기간": row.get("period_start"), "저장 경로": raw_path or "없음",
        "수집 시각": row.get("collected_at"), "체크섬": (row.get("checksum") or "")[:16],
        "크기(Byte)": row.get("byte_count"), "상태": row.get("collection_status"),
        "파일 확인": "존재" if exists else "누락 파일" if raw_path else "파일 없음",
    })
show_table(summary_rows, title="수집 실행 요약표", limit=20)
show_table(file_rows, title="원천별 파일 목록표", limit=20)

,상태,시도 수
0,success,19
1,duplicate,82
2,failed,0


,원천,자료유형,기간 시작,기간 종료,pjseq,주차,상태,HTTP,형식,행·레코드 수,실패 사유
0,internet_response_weekly,인터넷반응DB_202647,202647,202647,NaN,None,success,200,xlsx,158,
1,total_response_monthly,종합반응DB_202610,202610,202610,NaN,None,success,200,xlsx,192,
2,comment_analysis,comments_rank,202646,202646,11487.0,202646,success,200,json,20,
3,comment_analysis,positive_comments_rank,202646,202646,11487.0,202646,success,200,json,20,
4,comment_analysis,sentiment_trend,202646,202646,11487.0,202646,success,200,json,14,
5,comment_analysis,sentiment_distribution,202646,202646,11487.0,202646,success,200,json,1,
6,comment_analysis,wordcloud,202646,202646,11487.0,202646,success,200,json,200,
7,comment_analysis,network_rank,202646,202646,11487.0,202646,success,200,json,10,
8,comment_analysis,network_edges,202646,202646,11487.0,202646,success,200,json,42,
9,comment_analysis,network_nodes,202646,202646,11487.0,202646,success,200,json,7,


,원천,자료유형,기간,저장 경로,수집 시각,체크섬,크기(Byte),상태,파일 확인
0,internet_response_weekly,인터넷반응DB_202647,202647,데이터\원본\인터넷반응_주간\인터넷반응DB_202647_c59098d02ffd.xlsx,2026-11-28T01:10:08+09:00,c59098d02ffd3964,25303,success,존재
1,total_response_monthly,종합반응DB_202610,202610,데이터\원본\종합반응_월간\종합반응DB_202610_ddb97d71cd89.xlsx,2026-11-28T01:10:08+09:00,ddb97d71cd89075f,43304,success,존재
2,comment_analysis,comments_rank,202646,데이터\원본\댓글반응_상세\comments_rank_ebd6f1375b99.json,2026-11-28T01:10:08+09:00,ebd6f1375b99b715,1256,success,존재
3,comment_analysis,positive_comments_rank,202646,데이터\원본\댓글반응_상세\positive_comments_rank_445b0fa0...,2026-11-28T01:10:08+09:00,445b0fa0238e0f50,1246,success,존재
4,comment_analysis,sentiment_trend,202646,데이터\원본\댓글반응_상세\sentiment_trend_f71dece5c7bd.json,2026-11-28T01:10:09+09:00,f71dece5c7bd81a1,1248,success,존재
5,comment_analysis,sentiment_distribution,202646,데이터\원본\댓글반응_상세\sentiment_distribution_10b5a956...,2026-11-28T01:10:09+09:00,10b5a956873f28d7,90,success,존재
6,comment_analysis,wordcloud,202646,데이터\원본\댓글반응_상세\wordcloud_b295b4847864.json,2026-11-28T01:10:09+09:00,b295b48478648e5f,19083,success,존재
7,comment_analysis,network_rank,202646,데이터\원본\댓글반응_상세\network_rank_f78ab649f31d.json,2026-11-28T01:10:09+09:00,f78ab649f31dfff5,505,success,존재
8,comment_analysis,network_edges,202646,데이터\원본\댓글반응_상세\network_edges_135fcdcb8b5b.json,2026-11-28T01:10:10+09:00,135fcdcb8b5b6eec,2582,success,존재
9,comment_analysis,network_nodes,202646,데이터\원본\댓글반응_상세\network_nodes_86edf96b38e7.json,2026-11-28T01:10:10+09:00,86edf96b38e7c6d2,311,success,존재


,원천,자료유형,기간,저장 경로,수집 시각,체크섬,크기(Byte),상태,파일 확인
0,internet_response_weekly,인터넷반응DB_202647,202647,데이터\원본\인터넷반응_주간\인터넷반응DB_202647_c59098d02ffd.xlsx,2026-11-28T01:10:08+09:00,c59098d02ffd3964,25303,success,존재
1,total_response_monthly,종합반응DB_202610,202610,데이터\원본\종합반응_월간\종합반응DB_202610_ddb97d71cd89.xlsx,2026-11-28T01:10:08+09:00,ddb97d71cd89075f,43304,success,존재
2,comment_analysis,comments_rank,202646,데이터\원본\댓글반응_상세\comments_rank_ebd6f1375b99.json,2026-11-28T01:10:08+09:00,ebd6f1375b99b715,1256,success,존재
3,comment_analysis,positive_comments_rank,202646,데이터\원본\댓글반응_상세\positive_comments_rank_445b0fa0...,2026-11-28T01:10:08+09:00,445b0fa0238e0f50,1246,success,존재
4,comment_analysis,sentiment_trend,202646,데이터\원본\댓글반응_상세\sentiment_trend_f71dece5c7bd.json,2026-11-28T01:10:09+09:00,f71dece5c7bd81a1,1248,success,존재
...,...,...,...,...,...,...,...,...,...
96,comment_analysis,sentiment_distribution,202646,데이터\원본\댓글반응_상세\sentiment_distribution_10b5a956...,2026-11-28T21:05:33+09:00,10b5a956873f28d7,90,duplicate,존재
97,comment_analysis,wordcloud,202646,데이터\원본\댓글반응_상세\wordcloud_bcd1a9c67ecd.json,2026-11-28T21:05:33+09:00,bcd1a9c67ecdead9,19083,duplicate,존재
98,comment_analysis,network_rank,202646,데이터\원본\댓글반응_상세\network_rank_f78ab649f31d.json,2026-11-28T21:05:34+09:00,f78ab649f31dfff5,505,duplicate,존재
99,comment_analysis,network_edges,202646,데이터\원본\댓글반응_상세\network_edges_135fcdcb8b5b.json,2026-11-28T21:05:34+09:00,135fcdcb8b5b6eec,2582,duplicate,존재


In [4]:
# 목적: 주간·월간 기대 기간과 실제 확보 상태를 분리해 확인한다.
# 입력: 실행 환경변수의 시작일·종료일과 수집 메타데이터.
# 출력: 주간·월간 커버리지표 및 확보·누락 수.
# 기간: 명시 기간이 없으면 이력에서 관측된 기간만 표시하며 연간 커버리지를 주장하지 않는다.
# NULL/0 처리: 관측되지 않은 기간은 미수집으로 표기하며 반응량 0으로 해석하지 않는다.
def week_codes(start, end):
    current = start - timedelta(days=start.weekday())
    values = []
    while current <= end:
        year, week, _ = current.isocalendar()
        values.append(f"{year}{week:02d}")
        current += timedelta(days=7)
    return values

def month_codes(start, end):
    current = start.replace(day=1)
    last = end.replace(day=1)
    values = []
    while current <= last:
        values.append(current.strftime("%Y%m"))
        current = (current.replace(day=28) + timedelta(days=4)).replace(day=1)
    return values

weekly = [row for row in metadata_rows if row.get("source_name") == "internet_response_weekly"]
monthly = [row for row in metadata_rows if row.get("source_name") == "total_response_monthly"]
weekly_codes_actual = sorted({str(row.get("period_start")) for row in weekly if row.get("period_start")})
monthly_codes_actual = sorted({str(row.get("period_start")) for row in monthly if row.get("period_start")})

if START_DATE and END_DATE:
    expected_weeks = week_codes(datetime.strptime(START_DATE, "%Y-%m-%d").date(), datetime.strptime(END_DATE, "%Y-%m-%d").date())
    expected_months = month_codes(datetime.strptime(START_DATE, "%Y-%m-%d").date(), datetime.strptime(END_DATE, "%Y-%m-%d").date())
    coverage_note = f"명시 기간 {START_DATE}~{END_DATE}"
else:
    expected_weeks, expected_months = weekly_codes_actual, monthly_codes_actual
    coverage_note = "시작일·종료일 미설정: 현재 이력에 관측된 기간만 표시"

def coverage_rows(codes, rows, label):
    output = []
    for value in codes:
        matched = [row for row in rows if str(row.get("period_start")) == value]
        statuses = {row.get("collection_status") for row in matched}
        has_file = any((ROOT / Path(str(row.get("file_path", "")).replace("\\", "/"))).is_file() for row in matched if row.get("file_path"))
        output.append({"기간 코드": value, "원본": label, "이력 건수": len(matched), "파일": "확보" if has_file else "미수집", "상태": ", ".join(sorted(statuses)) or "미수집", "누락 사유": "" if matched else "메타데이터 없음"})
    return output

display(HTML(f"<p style='color:#4B5563'>커버리지 기준: {coverage_note}</p>"))
show_table(coverage_rows(expected_weeks, weekly, "인터넷반응DB 주간 XLSX"), title="주간 기간 커버리지표", limit=60)
show_table(coverage_rows(expected_months, monthly, "종합반응DB 월간 XLSX"), title="월간 기간 커버리지표", limit=24)

,기간 코드,원본,이력 건수,파일,상태,누락 사유
0,202647,인터넷반응DB 주간 XLSX,11,확보,"duplicate, success",


,기간 코드,원본,이력 건수,파일,상태,누락 사유
0,202610,종합반응DB 월간 XLSX,10,확보,"duplicate, success",


,기간 코드,원본,이력 건수,파일,상태,누락 사유
0,202610,종합반응DB 월간 XLSX,10,확보,"duplicate, success",


## 4. 실제 응답 형식과 구조

확장자만 믿지 않고 이슈 #3 수집기가 확인한 응답 형식과 구조 요약을 표시한다. 아래 XLSX 표는 읽기 전용으로 확인하며 원본을 변경하지 않는다.

In [5]:
# 목적: 메타데이터와 실제 원본에서 XLSX·JSON 구조를 확인한다.
# 입력: 성공 또는 중복 수집 이력과 원본 XLSX·JSON 파일.
# 출력: 형식·구조 요약, Excel 시트·헤더, JSON 키 구조표.
# 기간: 각 파일의 메타데이터 기간과 식별자를 사용한다.
# NULL/0 처리: JSON NULL·빈 배열을 구조로 표시하고 값 0으로 바꾸지 않는다.
structure_rows, excel_rows, json_rows = [], [], []
seen = set()
for row in metadata_rows:
    path_value = row.get("file_path")
    key = (row.get("dataset_name"), row.get("checksum"))
    if key in seen or not path_value:
        continue
    seen.add(key)
    path = ROOT / Path(str(path_value).replace("\\", "/"))
    summary = row.get("structure_summary") or {}
    exists = path.is_file()
    structure_rows.append({"원천": row.get("source_name"), "자료유형": row.get("dataset_name"), "형식": row.get("response_format"), "구조": summary.get("top_level") or f"시트 {summary.get('workbook_sheet_count', 0)}개", "행·레코드 수": row.get("row_count"), "기간·프로그램": f"{row.get('period_start') or ''} / {row.get('request_pjseq') or ''}", "파일": "존재" if exists else "누락"})
    if row.get("response_format") == "xlsx" and exists:
        workbook = load_workbook(path, read_only=True, data_only=False)
        for sheet in workbook.worksheets:
            values = list(sheet.iter_rows(min_row=1, max_row=min(8, sheet.max_row), values_only=True))
            header_rows = [" | ".join("" if value is None else str(value) for value in value_row) for value_row in values]
            excel_rows.append({"자료유형": row.get("dataset_name"), "시트": sheet.title, "물리 행": sheet.max_row, "물리 열": sheet.max_column, "상단 원본 행": " / ".join(header_rows[:3]), "병합 셀": summary.get("sheets", [{}])[0].get("merged_cell_count", "메타데이터 없음")})
        workbook.close()
    if row.get("response_format") == "json":
        json_rows.append({"자료유형": row.get("dataset_name"), "최상위": summary.get("top_level"), "주요 키": ", ".join(summary.get("fields", [])), "NULL 필드": ", ".join(summary.get("null_fields", [])) or "없음", "레코드 수": row.get("row_count"), "요청 식별자": f"pjseq={row.get('request_pjseq')}, nweek={row.get('request_nweek')}", "페이지네이션": "미제공 확인 필요"})
show_table(structure_rows, title="원천별 응답 형식·구조 요약표", limit=20)
show_table(excel_rows, title="Excel 시트·컬럼 구조표", limit=10)
show_table(json_rows, title="JSON 레코드·키 구조표", limit=20)

C:\Users\yooha\OneDrive\Desktop\git-hub-projects\Racoi\BrandCast\.venv\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


,원천,자료유형,형식,구조,행·레코드 수,기간·프로그램,파일
0,internet_response_weekly,인터넷반응DB_202647,xlsx,시트 1개,158,202647 /,존재
1,total_response_monthly,종합반응DB_202610,xlsx,시트 1개,192,202610 /,존재
2,comment_analysis,comments_rank,json,list,20,202646 / 11487,존재
3,comment_analysis,positive_comments_rank,json,list,20,202646 / 11487,존재
4,comment_analysis,sentiment_trend,json,list,14,202646 / 11487,존재
5,comment_analysis,sentiment_distribution,json,list,1,202646 / 11487,존재
6,comment_analysis,wordcloud,json,list,200,202646 / 11487,존재
7,comment_analysis,network_rank,json,list,10,202646 / 11487,존재
8,comment_analysis,network_edges,json,list,42,202646 / 11487,존재
9,comment_analysis,network_nodes,json,list,7,202646 / 11487,존재


,자료유형,시트,물리 행,물리 열,상단 원본 행,병합 셀
0,인터넷반응DB_202647,인터넷반응DB,1,1,,10
1,종합반응DB_202610,종합반응DB,1,1,,17
2,인터넷반응DB_202647,인터넷반응DB,1,1,,10
3,종합반응DB_202610,종합반응DB,1,1,,17
4,인터넷반응DB_202647,인터넷반응DB,1,1,,10
5,종합반응DB_202610,종합반응DB,1,1,,17
6,인터넷반응DB_202647,인터넷반응DB,1,1,,10
7,종합반응DB_202610,종합반응DB,1,1,,17
8,인터넷반응DB_202647,인터넷반응DB,1,1,,10
9,종합반응DB_202610,종합반응DB,1,1,,17


,자료유형,최상위,주요 키,NULL 필드,레코드 수,요청 식별자,페이지네이션
0,comments_rank,list,"frequency, keyword, nweek, rk1",없음,20,"pjseq=11487, nweek=202646",미제공 확인 필요
1,positive_comments_rank,list,"frequency, keyword, nweek, rk1",없음,20,"pjseq=11487, nweek=202646",미제공 확인 필요
2,sentiment_trend,list,"negativescore, nweek, positivescore",없음,14,"pjseq=11487, nweek=202646",미제공 확인 필요
3,sentiment_distribution,list,"negativescore, nweek, positivescore",없음,1,"pjseq=11487, nweek=202646",미제공 확인 필요
4,wordcloud,list,"frequency, keyword, nweek, programid, sentimen...",없음,200,"pjseq=11487, nweek=202646",미제공 확인 필요
5,network_rank,list,"bigram, frequency, rk1",없음,10,"pjseq=11487, nweek=202646",미제공 확인 필요
6,network_edges,list,"bigram, frequency, rk1, target",없음,42,"pjseq=11487, nweek=202646",미제공 확인 필요
7,network_nodes,list,"bigram, frequency, rk1",없음,7,"pjseq=11487, nweek=202646",미제공 확인 필요
8,wordcloud,list,"frequency, keyword, nweek, programid, sentimen...",없음,200,"pjseq=11487, nweek=202646",미제공 확인 필요
9,wordcloud,list,"frequency, keyword, nweek, programid, sentimen...",없음,200,"pjseq=11487, nweek=202646",미제공 확인 필요


,자료유형,최상위,주요 키,NULL 필드,레코드 수,요청 식별자,페이지네이션
0,comments_rank,list,"frequency, keyword, nweek, rk1",없음,20,"pjseq=11487, nweek=202646",미제공 확인 필요
1,positive_comments_rank,list,"frequency, keyword, nweek, rk1",없음,20,"pjseq=11487, nweek=202646",미제공 확인 필요
2,sentiment_trend,list,"negativescore, nweek, positivescore",없음,14,"pjseq=11487, nweek=202646",미제공 확인 필요
3,sentiment_distribution,list,"negativescore, nweek, positivescore",없음,1,"pjseq=11487, nweek=202646",미제공 확인 필요
4,wordcloud,list,"frequency, keyword, nweek, programid, sentimen...",없음,200,"pjseq=11487, nweek=202646",미제공 확인 필요
5,network_rank,list,"bigram, frequency, rk1",없음,10,"pjseq=11487, nweek=202646",미제공 확인 필요
6,network_edges,list,"bigram, frequency, rk1, target",없음,42,"pjseq=11487, nweek=202646",미제공 확인 필요
7,network_nodes,list,"bigram, frequency, rk1",없음,7,"pjseq=11487, nweek=202646",미제공 확인 필요
8,wordcloud,list,"frequency, keyword, nweek, programid, sentimen...",없음,200,"pjseq=11487, nweek=202646",미제공 확인 필요
9,wordcloud,list,"frequency, keyword, nweek, programid, sentimen...",없음,200,"pjseq=11487, nweek=202646",미제공 확인 필요


In [6]:
# 목적: 댓글반응에서 실제 제공·미제공 범위를 자료유형으로 확인한다.
# 입력: 댓글반응 JSON 수집 이력의 dataset_name과 구조 요약.
# 출력: Q4·Q6 후속 검토를 위한 제공·미제공 항목표.
# 기간: 프로그램·주차별 공개 JSON의 실제 기록 기간을 사용한다.
# NULL/0 처리: 중립 감성·전체 댓글 표본 수는 파일에 없으면 미제공으로 표시한다.
comment_names = {row.get("dataset_name") for row in metadata_rows if row.get("source_name") == "comment_analysis"}
def provided(*names):
    return "제공" if any(name in comment_names for name in names) else "미제공"
provision_rows = [
    {"항목": "키워드", "제공 여부": provided("comments_rank", "wordcloud"), "원천 자료유형": "comments_rank, wordcloud", "후속 사용": "Q4 관심 주제 입력 검토", "제한": "원천별 의미와 중복 확인 필요"},
    {"항목": "긍정 키워드", "제공 여부": provided("positive_comments_rank"), "원천 자료유형": "positive_comments_rank", "후속 사용": "긍정 맥락 보조", "제한": "전체 감성 대표로 해석 금지"},
    {"항목": "연관어·연결 강도", "제공 여부": provided("network_rank", "network_edges", "network_nodes"), "원천 자료유형": "network_rank, network_edges, network_nodes", "후속 사용": "Q4 관계 보조", "제한": "의미 유사도 자체가 아님"},
    {"항목": "긍정·부정 비율", "제공 여부": provided("sentiment_trend", "sentiment_distribution"), "원천 자료유형": "sentiment_trend, sentiment_distribution", "후속 사용": "Q6 위험 검토", "제한": "제공값 정의 확인 필요"},
    {"항목": "중립 감성", "제공 여부": "미제공", "원천 자료유형": "없음", "후속 사용": "직접 산출 불가", "제한": "임의 생성 금지"},
    {"항목": "전체 댓글 표본 수", "제공 여부": "미제공", "원천 자료유형": "없음", "후속 사용": "충분성 판단", "제한": "워드클라우드 건수를 댓글 수로 표현 금지"},
]
show_table(provision_rows, title="댓글반응 제공·미제공 항목표", limit=10)

,항목,제공 여부,원천 자료유형,후속 사용,제한
0,키워드,제공,"comments_rank, wordcloud",Q4 관심 주제 입력 검토,원천별 의미와 중복 확인 필요
1,긍정 키워드,제공,positive_comments_rank,긍정 맥락 보조,전체 감성 대표로 해석 금지
2,연관어·연결 강도,제공,"network_rank, network_edges, network_nodes",Q4 관계 보조,의미 유사도 자체가 아님
3,긍정·부정 비율,제공,"sentiment_trend, sentiment_distribution",Q6 위험 검토,제공값 정의 확인 필요
4,중립 감성,미제공,없음,직접 산출 불가,임의 생성 금지
5,전체 댓글 표본 수,미제공,없음,충분성 판단,워드클라우드 건수를 댓글 수로 표현 금지


,항목,제공 여부,원천 자료유형,후속 사용,제한
0,키워드,제공,"comments_rank, wordcloud",Q4 관심 주제 입력 검토,원천별 의미와 중복 확인 필요
1,긍정 키워드,제공,positive_comments_rank,긍정 맥락 보조,전체 감성 대표로 해석 금지
2,연관어·연결 강도,제공,"network_rank, network_edges, network_nodes",Q4 관계 보조,의미 유사도 자체가 아님
3,긍정·부정 비율,제공,"sentiment_trend, sentiment_distribution",Q6 위험 검토,제공값 정의 확인 필요
4,중립 감성,미제공,없음,직접 산출 불가,임의 생성 금지
5,전체 댓글 표본 수,미제공,없음,충분성 판단,워드클라우드 건수를 댓글 수로 표현 금지


## 5. 원본 표본

표본은 원본 필드와 값을 변경하지 않고 최대 10행만 표시한다. 원문 댓글과 개인정보는 표시하지 않는다.

In [7]:
# 목적: 세 원천의 실제 원본 표본과 전체 행·레코드 수를 확인한다.
# 입력: 존재하는 XLSX·JSON 원본 파일.
# 출력: 인터넷반응, 종합반응, 댓글 키워드·감성·연관어 표본.
# 기간: 표본을 만든 파일의 메타데이터 기간을 함께 표시한다.
# NULL/0 처리: 원본 없음·파싱 실패는 가상 행을 만들지 않고 상태로 표시한다.
def latest_existing(format_name, dataset_name=None):
    candidates = []
    for row in metadata_rows:
        if row.get("response_format") != format_name:
            continue
        if dataset_name and row.get("dataset_name") != dataset_name:
            continue
        value = row.get("file_path")
        path = ROOT / Path(str(value).replace("\\", "/")) if value else None
        if path and path.is_file():
            candidates.append((row, path))
    return candidates[-1] if candidates else (None, None)

def excel_sample(dataset_name, title):
    row, path = latest_existing("xlsx", dataset_name)
    if not path:
        show_table([], title=f"{title} — 원본 없음")
        return
    book = load_workbook(path, read_only=True, data_only=False)
    sheet = book.active
    records = list(sheet.iter_rows(min_row=1, max_row=min(10, sheet.max_row), values_only=True))
    book.close()
    show_table([{f"열{index + 1}": value for index, value in enumerate(record)} for record in records], title=f"{title} 표본 · 전체 물리 행 {sheet.max_row}", limit=10)

def json_sample(dataset_name, title):
    row, path = latest_existing("json", dataset_name)
    if not path:
        show_table([], title=f"{title} — 원본 없음")
        return
    value = json.loads(path.read_text(encoding="utf-8-sig"))
    rows = value if isinstance(value, list) else [value]
    show_table(rows, title=f"{title} 표본 · 전체 레코드 {len(rows)} · pjseq={row.get('request_pjseq')} · nweek={row.get('request_nweek')}", limit=10)

excel_sample("인터넷반응DB_202647", "인터넷반응 원본")
excel_sample("종합반응DB_202610", "종합반응 원본")
json_sample("wordcloud", "댓글 키워드·워드클라우드")
json_sample("sentiment_distribution", "댓글 감성 분포")
json_sample("network_edges", "댓글 연관어")

C:\Users\yooha\OneDrive\Desktop\git-hub-projects\Racoi\BrandCast\.venv\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


,열1
0,None


C:\Users\yooha\OneDrive\Desktop\git-hub-projects\Racoi\BrandCast\.venv\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


,열1
0,None


,sentimenttypecode,nweek,keyword,programid,frequency
0,2,202646,울다,11487,25
1,3,202646,바뀌다,11487,25
2,3,202646,짚다,11487,25
3,3,202646,끌리다,11487,24
4,2,202646,싸우다,11487,24
5,2,202646,아프다,11487,24
6,3,202646,정하다,11487,23
7,3,202646,바꾸다,11487,23
8,3,202646,갖다,11487,23
9,3,202646,내리다,11487,22


,nweek,positivescore,negativescore
0,202646,0.502718,0.497282


,rk1,bigram,target,frequency
0,1,영호,옥순,223
1,2,영호,현숙,145
2,3,현숙,옥순,130
3,4,현숙,영호,90
4,5,현숙,영철,86
5,6,순자,옥순,85
6,7,영호,영철,80
7,8,영철,현숙,78
8,9,영철,영호,76
9,10,광수,정숙,70


## 6. MariaDB 적재 전달 기준과 결론

| 원본 | 이슈 #5 적재 단위 | 필수 출처 식별자 |
|---|---|---|
| 인터넷반응 XLSX | 실제 데이터 행 1개 | 파일 체크섬, 시트, 원천 행 번호, 주차 |
| 종합반응 XLSX | 실제 데이터 행 1개 | 파일 체크섬, 시트, 원천 행 번호, 월 |
| 댓글반응 JSON | 자료유형별 반복 레코드 1개 | 파일 체크섬, `dataset_name`, `pjseq`, `nweek`, 배열 위치 |
| 수집 메타데이터 | 다운로드 시도 1회 | URL, 수집 시각, 기간, 상태, 체크섬, 파일 경로 |

이 노트북은 실제 파일과 JSONL 이력을 기준으로 확보 범위와 구조를 보여준다. 파일이 없는 이력·실패·미제공 필드는 숨기지 않으며, 수집되지 않은 기간이나 제공되지 않은 감성 항목을 0 또는 추정값으로 바꾸지 않는다. MariaDB 적재와 품질검사는 이슈 #5에서 수행한다.